In [ ]:
import pandas as pd
import plotly.graph_objects as go

lab_to_class = list(pd.read_csv("../../datasets/imagenet/classes.txt", header=None).values[0])

In [99]:
class_counts = pd.read_csv("../../experiment_data/feature_counts_imagenet/e-4V10H99.csv").iloc[:, 1:]
class_counts = class_counts.rename(columns={"Count Majority": "maj", "Class": "cls", "Count": "val"})
class_counts["lab"] = class_counts["cls"].map(lambda x: lab_to_class.index(x))
class_counts.head()

,cls,val,maj,lab
0,tench,1,1,0
1,goldfish,1,1,1
2,great white shark,1,1,2
3,tiger shark,1,1,3
4,hammerhead,0,0,4


In [106]:
coverage = pd.read_csv("../../experiment_data/feature_counts_imagenet/e-4V10H99Coverage.csv").iloc[:, 1:]
coverage = coverage.rename(columns={"Coverage": "cov", "True Class": "cls", "True Label": "lab", "Proportion": "prop", "Count": "size"})
coverage = coverage.sort_values("lab")
coverage.head()

,lab,cls,count,prop,cov
98,0,tench,658,0.002829,0.4874
152,1,goldfish,562,0.002417,0.4163
553,2,great white shark,62,0.000267,0.0459
626,3,tiger shark,23,0.000099,0.0170
588,5,electric ray,40,0.000172,0.0296


In [103]:
valleys = pd.read_csv("../../experiment_data/feature_counts_imagenet/e-4V10H99Valleys.csv").iloc[:, 1:]
valleys = valleys.rename(columns={"major class size": "majsize", "majority class": "cls", "major class coverage": "majcov"}).drop(columns=["idx"])
valleys["lab"] = valleys["cls"].map(lambda x: lab_to_class.index(x))
valleys = valleys.sort_values("lab")
valleys.head()

,id,fstart,fend,ftype,pers,volume,logvol,cls,homogeneity,majsize,majcov,lab
720,11736,2.384186e-07,0.002685,minima-saddle,0.002685,658,6.490724,tench,1.0,658,0.4874,0
191,1680,-0.000000e+00,0.000520,minima-saddle,0.000476,562,6.333280,goldfish,1.0,562,0.4163,1
100,688,4.339124e-05,0.000719,minima-saddle,0.000676,62,4.143135,great white shark,1.0,62,0.0459,2
371,4231,1.144403e-05,0.000532,minima-saddle,0.000471,23,3.178054,tiger shark,1.0,23,0.0170,3
114,765,1.454343e-05,0.000535,minima-saddle,0.000255,40,3.713572,electric ray,1.0,40,0.0296,5


How many classes have corresponding homogeneous valleys? 

In [ ]:
homo_val_owners = (class_counts["maj"] != 0).value_counts()[True]
homo_val_owners

np.int64(724)

In [ ]:
homo_thresh = 0.99
valleys_homo = valleys[valleys["homogeneity"] >= homo_thresh]

How many valleys are homogeneous?

In [142]:
valleys_homo_count = len(valleys_homo)
print(f"{valleys_homo_count} / {len(valleys)}")

751 / 751


How many classes have at least 10% coverage in their homogeneous valleys?

In [146]:
valleys_homo.head()

,id,fstart,fend,ftype,pers,volume,logvol,cls,homogeneity,majsize,majcov,lab
720,11736,2.384186e-07,0.002685,minima-saddle,0.002685,658,6.490724,tench,1.0,658,0.4874,0
191,1680,-0.000000e+00,0.000520,minima-saddle,0.000476,562,6.333280,goldfish,1.0,562,0.4163,1
100,688,4.339124e-05,0.000719,minima-saddle,0.000676,62,4.143135,great white shark,1.0,62,0.0459,2
371,4231,1.144403e-05,0.000532,minima-saddle,0.000471,23,3.178054,tiger shark,1.0,23,0.0170,3
114,765,1.454343e-05,0.000535,minima-saddle,0.000255,40,3.713572,electric ray,1.0,40,0.0296,5


In [152]:
valleys_homo["lab"].value_counts()

lab
502    2
444    2
518    2
174    2
109    2
      ..
991    1
993    1
994    1
995    1
1      1
Name: count, Length: 724, dtype: int64

In [220]:
pooled_cov = valleys_homo.groupby("lab").aggregate(majcov=('majcov', 'sum'), size=('majcov', 'size'))
maj_cov = (pooled_cov["majcov"] >= 0.1).value_counts()[True]

maj_cov, 1000 - maj_cov

(np.int64(476), np.int64(524))